# AIS Distance Prediction Training

Train an image-based position predictor from the Hugging Face `vision_offset_dataset`.
Data is expected under `ws_aic/data`, and checkpoints are saved under `ws_aic/model`.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR

PosixPath('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction')

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from model import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_WEIGHT_ROOT,
    VisionOffsetDataset,
    build_resnet_position_model,
    compute_target_stats,
    download_vision_offset_dataset,
    evaluate,
    fit,
    load_samples,
    seed_everything,
    split_samples_by_stratified_group_kfold,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

/media/hdd0/whyz/delivery-delay/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
DATASET_ROOT = DEFAULT_DATASET_ROOT
WEIGHT_ROOT = DEFAULT_WEIGHT_ROOT

CAMERAS = ('left', 'center', 'right')
TARGET_KEYS = ('x_mm', 'y_mm', 'z_mm')
IMAGE_SIZE = (256, 288)
BATCH_SIZE = 8
NUM_WORKERS = 8
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'resnet50'
PRETRAINED = True
AGGREGATION = 'concat'
N_SPLITS = 5
FOLD_INDEX = 0
STRATIFY_KEYS = ('task_type', 'port_type')
RUN_NAME = f"vision_offset_{BACKBONE}_{'_'.join(CAMERAS)}_{AGGREGATION}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME

(PosixPath('/home/whyz/aic_sejong/ws_aic/data/aic-entrance-dataset/v1.2/vision_offset_dataset'),
 PosixPath('/home/whyz/aic_sejong/ws_aic/model/ais_distance_prediction/vision_offset_resnet50_left_center_right_concat'))

In [4]:
DATASET_ROOT = download_vision_offset_dataset(max_workers=8)

samples = load_samples(DATASET_ROOT)
train_samples, val_samples, test_samples = split_samples_by_stratified_group_kfold(
    samples,
    n_splits=N_SPLITS,
    fold_index=FOLD_INDEX,
    seed=42,
    stratify_keys=STRATIFY_KEYS,
)
target_stats = compute_target_stats(train_samples, TARGET_KEYS)

print(f"samples: train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}")
print('target mean:', target_stats['mean'].tolist())
print('target std :', target_stats['std'].tolist())

samples: train=4480 val=1120 test=0
target mean: [-0.060394033789634705, 1.2092254161834717, -19.659921646118164]
target std : [15.510475158691406, 5.5646233558654785, 16.06197166442871]


In [5]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=train_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=eval_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['raw_target'][:3]

(torch.Size([8, 3, 3, 256, 288]),
 torch.Size([8, 3]),
 tensor([[-4.7195e+00, -1.6081e-01, -1.5337e+01],
         [-2.6292e+01, -1.8328e+00, -1.5752e+01],
         [-8.9369e-03,  1.4217e+00, -1.7677e+01]]))

In [7]:
model = build_resnet_position_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=len(TARGET_KEYS),
    hidden_dim=256,
    dropout=0.1,
    aggregation=AGGREGATION,
    num_views=len(CAMERAS),
)
model.to(device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)

26720838

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'cameras': CAMERAS,
    'target_keys': TARGET_KEYS,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
    'aggregation': AGGREGATION,
    'num_views': len(CAMERAS),
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    config=config,
)
history[-1]

epochs:   5%|▌         | 1/20 [01:28<28:04, 88.68s/it, train_loss=0.0696, val_euclidean=5.642, val_loss=0.0870]

epoch=001 train_loss=0.0696 val_loss=0.0870 val_mae=2.726 val_euclidean=5.642


epochs:  10%|█         | 2/20 [02:55<26:15, 87.54s/it, train_loss=0.0238, val_euclidean=4.845, val_loss=0.0664]

epoch=002 train_loss=0.0238 val_loss=0.0664 val_mae=2.429 val_euclidean=4.845


epochs:  15%|█▌        | 3/20 [04:22<24:41, 87.18s/it, train_loss=0.0186, val_euclidean=4.706, val_loss=0.0682]

...

epochs:  95%|█████████▌| 19/20 [27:30<01:26, 86.73s/it, train_loss=0.0062, val_euclidean=4.537, val_loss=0.0655]

epoch=019 train_loss=0.0062 val_loss=0.0655 val_mae=2.206 val_euclidean=4.537


epochs: 100%|██████████| 20/20 [28:57<00:00, 86.87s/it, train_loss=0.0064, val_euclidean=4.221, val_loss=0.0621]

epoch=020 train_loss=0.0064 val_loss=0.0621 val_mae=2.105 val_euclidean=4.221


{'epoch': 20.0,
 'train_loss': 0.006416124085990305,
 'val_loss': 0.06214242962305434,
 'val_mae': 2.1051974296569824,
 'val_rmse': 3.9823219776153564,
 'val_euclidean': 4.2213921546936035}

In [9]:
val_metrics = evaluate(
    model,
    val_loader,
    device,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_metrics

{'loss': 0.06214242962305434,
 'mae': 2.1051974296569824,
 'rmse': 3.9823219776153564,
 'euclidean': 4.2213921546936035}

In [10]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    pred = model(batch['image'].to(device)).cpu()
pred_mm = pred * target_stats['std'] + target_stats['mean']
preview = torch.cat([pred_mm[:8], batch['raw_target'][:8]], dim=1)
print('columns: pred_x pred_y pred_z target_x target_y target_z')
display(preview)

correct_count = 0
total_count = 0
before_distances = []
after_distances = []

with torch.inference_mode():
    for eval_batch in val_loader:
        pred = model(eval_batch['image'].to(device)).cpu()
        pred_mm = pred * target_stats['std'] + target_stats['mean']
        target_mm = eval_batch['raw_target']

        before_distance = torch.linalg.vector_norm(target_mm, dim=1)
        after_distance = torch.linalg.vector_norm(target_mm - pred_mm, dim=1)
        improved = after_distance < before_distance

        correct_count += int(improved.sum())
        total_count += int(improved.numel())
        before_distances.append(before_distance)
        after_distances.append(after_distance)

before_distance_mm = torch.cat(before_distances)
after_distance_mm = torch.cat(after_distances)
improvement_metrics = {
    'criterion': 'after_distance_mm < before_distance_mm',
    'accuracy': correct_count / max(total_count, 1),
    'correct': correct_count,
    'total': total_count,
    'before_distance_mean_mm': float(before_distance_mm.mean()),
    'after_distance_mean_mm': float(after_distance_mm.mean()),
    'distance_improvement_mean_mm': float((before_distance_mm - after_distance_mm).mean()),
}
improvement_metrics

columns: pred_x pred_y pred_z target_x target_y target_z


tensor([[-1.3065e+00,  2.1042e-01, -1.7230e+02,  4.8630e-01, -7.4137e-01,
         -1.4671e+02],
        [-1.3065e+00,  2.1042e-01, -1.7230e+02,  2.3686e+01, -7.4140e-01,
         -1.4671e+02],
        [-4.5970e-01, -6.2671e-02, -1.6951e+02,  2.3643e+00, -8.7939e-01,
         -1.3657e+02],
        [-4.5970e-01, -6.2671e-02, -1.6951e+02,  2.5564e+01, -8.7942e-01,
         -1.3657e+02],
        [ 5.3969e+00,  8.3588e-01, -1.2765e+02,  5.3512e+00,  1.2327e+00,
         -1.1164e+02],
        [ 5.3969e+00,  8.3588e-01, -1.2765e+02,  2.8551e+01,  1.2326e+00,
         -1.1164e+02],
        [ 7.6644e+00,  2.5791e+00, -9.7621e+01,  8.8747e+00,  3.3418e+00,
         -8.3394e+01],
        [ 7.6644e+00,  2.5791e+00, -9.7621e+01,  3.2075e+01,  3.3418e+00,
         -8.3394e+01]])

{'criterion': 'after_distance_mm < before_distance_mm',
 'accuracy': 0.8588541666666667,
 'correct': 1649,
 'total': 1920,
 'before_distance_mean_mm': 25.248506546020508,
 'after_distance_mean_mm': 13.077256202697754,
 'distance_improvement_mean_mm': 12.171249389648438}